In [0]:
%sql
-- 01_bronze.sql -- Bronze layer: append-only landing of raw Chess.com snapshots.
-- Run in a Databricks SQL notebook on Serverless compute.
--
-- Design: bronze mirrors the source's shape -- one row per landed snapshot
-- file, with that month's `games` array intact. Grain normalization (one row
-- per game, one row per move) happens in silver. COPY INTO tracks which files
-- it has already ingested, so re-running this notebook is a no-op until new
-- snapshot files land in S3. That property is what makes bronze safely
-- append-only on a schedule.

-- ---------------------------------------------------------------
-- 0. Namespaces: the medallion layers as schemas.
--    If CREATE CATALOG errors on Free Edition, fall back to the built-in
--    catalog: replace `chess.bronze` with `workspace.chess_bronze` (etc.)
--    throughout the project.
-- ---------------------------------------------------------------
CREATE CATALOG IF NOT EXISTS chess;
CREATE SCHEMA IF NOT EXISTS chess.bronze;
CREATE SCHEMA IF NOT EXISTS chess.silver;
CREATE SCHEMA IF NOT EXISTS chess.gold;

-- ---------------------------------------------------------------
-- 1. Bronze table. Created empty; COPY INTO infers and evolves the
--    schema (mergeSchema) as snapshot files arrive.
-- ---------------------------------------------------------------
CREATE TABLE IF NOT EXISTS chess.bronze.games_raw;

-- ---------------------------------------------------------------
-- 2. Idempotent append of any not-yet-loaded snapshot files.
--    _metadata columns preserve provenance: which file, when modified,
--    when ingested. The Hive-style path (username=/year=/month=) rides
--    along inside _source_file for silver to extract.
-- ---------------------------------------------------------------
COPY INTO chess.bronze.games_raw
FROM (
  SELECT
    *,
    _metadata.file_path              AS _source_file,
    _metadata.file_modification_time AS _file_modified_at,
    current_timestamp()              AS _ingested_at
  FROM 's3://chess-pipeline-raw-jhr/raw/chesscom/games/'
)
FILEFORMAT = JSON
FORMAT_OPTIONS ('multiLine' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');

-- ---------------------------------------------------------------
-- 3. Verify: one row per snapshot file.
--    Expect 52 files (48 backfill + 4 timestamped re-pulls) and roughly
--    6,000 game rows including duplicates.
-- ---------------------------------------------------------------
SELECT
  count(*)         AS snapshot_files,
  sum(size(games)) AS game_rows_incl_duplicates
FROM chess.bronze.games_raw;

-- ---------------------------------------------------------------
-- 4. The dedup preview: bronze intentionally holds the same game in
--    multiple snapshots (trailing-month re-pulls). Distinct uuids is
--    what silver will hold after MERGE.
-- ---------------------------------------------------------------
SELECT
  count(g.uuid)          AS game_rows_incl_duplicates,
  count(DISTINCT g.uuid) AS distinct_games
FROM chess.bronze.games_raw
LATERAL VIEW explode(games) exploded AS g;

-- ---------------------------------------------------------------
-- 5. Eyeball one game object to see the raw fields silver will type:
--    uuid, url, end_time, rated, time_class, time_control, rules,
--    white/black {username, rating, result}, pgn.
-- ---------------------------------------------------------------
SELECT g.*
FROM chess.bronze.games_raw
LATERAL VIEW explode(games) exploded AS g
LIMIT 1;